# Building a Code Assistant with a Decoder-Only Transformer

In this notebook, we transform our decoder-only transformer into an **Instruction-Following Coding Assistant**.

### Key Differences from Standard Next-Token Prediction:
1. **Supervised Instruction Tuning (SFT):** The model is trained on `(Instruction, Code)` pairs rather than continuous raw text.
2. **Unified Prompt Template:** Every example follows a clean format:
   ```text
   ### Instruction:
   Write a python program to find the maximum number in an array.

   ### Response:
   def find_max(arr):
       return max(arr)
   ```
3. **Code-Optimized Byte-Level BPE Tokenizer:** Preserves Python syntax, indentation (tabs/spaces), and reserved keywords.
4. **Prompt / Loss Masking (`ignore_index=-100`):** We compute cross-entropy loss **only on the generated code tokens**, masking out the prompt tokens. The model focuses 100% of its learning capacity on writing correct code.
5. **Interactive Generation with EOS Stopping:** Prompt injection, autoregressive decoding (greedy / nucleus sampling), and early stopping at `[EOS]`.

In [ ]:
import os
import json
import math
import time
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Data Loading & Unification

Our workspace has multiple raw datasets:
- `../dataset/raw/train/python_code_instructions.parquet`

We unify all of these into standard dictionaries:
`{'prompt': '### Instruction:\n...\n\n### Response:\n', 'response': '<code>'}`

In [51]:
PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"
PROMPT_WITH_INPUT_TEMPLATE = "### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

def format_instruction(instruction, input_text=""):
    instruction = (instruction or "").strip()
    input_text = (input_text or "").strip()
    
    # Omit useless input placeholders
    if input_text and input_text.lower() not in ["", "none", "not applicable", "no input", "n/a"]:
        return PROMPT_WITH_INPUT_TEMPLATE.format(instruction=instruction, input_text=input_text)
    else:
        return PROMPT_TEMPLATE.format(instruction=instruction)

def format_response(code_str):
    """
    Ensures the response is cleanly formatted in a Markdown Python code block.
    """
    code_str = (code_str or "").strip()
    if not code_str.startswith("```"):
        return f"```python\n{code_str}\n```"
    return code_str

def load_unified_parquet_instructions(parquet_path, max_samples=None):
    """
    Loads Alpaca-style code instructions from Parquet (e.g. python_code_instructions.parquet)
    """
    import pandas as pd
    print(f"Loading {parquet_path}...")
    df = pd.read_parquet(parquet_path)
    unified_samples = []
    for _, row in df.iterrows():
        inst = str(row.get("instruction", "") or "").strip()
        inp = str(row.get("input", "") or "").strip()
        out = str(row.get("output", "") or "").strip()
        txt = str(row.get("prompt", "") or "").strip()
        
        if not inst or not out:
            continue
            
        prompt = format_instruction(inst, inp)
        formatted_code = format_response(out)
        unified_samples.append({
            "prompt": prompt,
            "response": formatted_code,
            "text": txt
        })
        
        if max_samples and len(unified_samples) >= max_samples:
            break
            
    print(f"Loaded {len(unified_samples)} unified samples from {parquet_path}")
    return unified_samples


In [ ]:
# Load the datasets
train_parquet_path = "../dataset/raw/train/python_code_instructions.parquet"

# 1. Load Parquet instructions (e.g. 35,000 samples for balanced training)
raw_dataset = load_unified_parquet_instructions(train_parquet_path)

# Display an example
print("\n" + "="*60)
print("SAMPLE UNIFIED RECORD:")
print("="*60)
sample = raw_dataset[0]
if sample.get("text"):
    print("--- [ORIGINAL 'text' FIELD FROM DATASET] ---\n" + sample["text"] + "\n")
print("--- [UNIFIED PROMPT] ---\n" + sample["prompt"])
print("--- [UNIFIED RESPONSE] ---\n" + sample["response"])

## 2. Dataset Splitting (90% Train / 5% Val / 5% Test)

We split our unified instruction dataset into 3 clean sets:
- **Train Set (90%):** Used to train model weights.
- **Validation Set (5%):** Used during training to monitor loss and save the best checkpoint.
- **Held-Out Test Set (5%):** Unseen questions with full prompt & response to evaluate final model performance.

In [ ]:
random.seed(SEED)
shuffled_indices = list(range(len(raw_dataset)))
random.shuffle(shuffled_indices)

n_total = len(raw_dataset)
val_size = int(n_total * 0.05)
test_size = int(n_total * 0.05)

val_indices = shuffled_indices[:val_size]
test_indices = shuffled_indices[val_size:val_size + test_size]
train_indices = shuffled_indices[val_size + test_size:]

train_data = [raw_dataset[i] for i in train_indices]
val_data   = [raw_dataset[i] for i in val_indices]
test_data  = [raw_dataset[i] for i in test_indices]

print(f"Train samples:      {len(train_data)} ({len(train_data)/n_total*100:.1f}%)")
print(f"Validation samples: {len(val_data)} ({len(val_data)/n_total*100:.1f}%)")
print(f"Held-Out Test:      {len(test_data)} ({len(test_data)/n_total*100:.1f}%)")

## 3. Byte-Level BPE Tokenizer for Code

Code requires a specialized vocabulary:
- Preserving exact indentation (tabs and sequences of spaces).
- Common keywords (`def`, `return`, `class`, `import`, `for`, `in`, `None`, `True`, `False`).
- Programming symbols (`==`, `!=`, `->`, `+=`, `[]`, `{}`, `()`, `__init__`).

We train a Byte-Level BPE tokenizer with special tokens:
- `[PAD]`: Padding token (id=0)
- `[UNK]`: Unknown token fallback
- `[BOS]`: Beginning of sequence
- `[EOS]`: End of sequence
- `[SEP]`: Separator

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, processors, normalizers

VOCAB_SIZE = 16000 # Compact vocabulary optimal for code & mini-GPT
TOKENIZER_PATH = "py_code_tokenizer.json"

# Check if code tokenizer already trained, otherwise train it
if os.path.exists(TOKENIZER_PATH):
    print(f"Loading existing code tokenizer from {TOKENIZER_PATH}...")
    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
else:
    print("Training new Byte-Level BPE Tokenizer for code...")
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]", byte_fallback=True))
    tokenizer.normalizer = normalizers.Sequence([normalizers.NFD()])
    # add_prefix_space=False preserves code indentation accurately
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    
    special_tokens = ["[PAD]", "[UNK]", "[BOS]", "[EOS]", "[SEP]"]
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=special_tokens,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
    )
    
    def code_iterator():
        for sample in train_data:
            yield sample["prompt"] + sample["response"]
            
    tokenizer.train_from_iterator(code_iterator(), trainer=trainer)
    tokenizer.decoder = decoders.ByteLevel()
    tokenizer.save(TOKENIZER_PATH)
    print(f"Tokenizer saved to {TOKENIZER_PATH}")

PAD_TOKEN_ID = tokenizer.token_to_id("[PAD]")
UNK_TOKEN_ID = tokenizer.token_to_id("[UNK]")
BOS_TOKEN_ID = tokenizer.token_to_id("[BOS]")
EOS_TOKEN_ID = tokenizer.token_to_id("[EOS]")

print(f"Vocab size: {tokenizer.get_vocab_size()}")
print(f"Special Token IDs -> PAD: {PAD_TOKEN_ID}, BOS: {BOS_TOKEN_ID}, EOS: {EOS_TOKEN_ID}")

In [ ]:
# Test tokenizer on a code snippet with indentation
test_code = "def find_max(arr):\n    return max(arr)"
encoded = tokenizer.encode(test_code)
print("Original text:\n", test_code)
print("\nEncoded token IDs:", encoded.ids)
print("Decoded text:\n", tokenizer.decode(encoded.ids))

## 4. Tokenization Function & The Crucial Loss Masking

### Why Loss Masking is Critical:
In standard language modeling, loss is calculated over every token.
If we calculate loss on the prompt (`### Instruction: Write a program...`), the model spends capacity predicting the user's question!

With **Loss Masking**:
- Prompt tokens are replaced with `-100` in the target labels.
- Response tokens remain as their token IDs.
- `nn.CrossEntropyLoss(ignore_index=-100)` ignores all prompt positions, calculating gradients **exclusively on the code response**.

In [ ]:
class CodeInstructionDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        
        eos_id = tokenizer.token_to_id("[EOS]")
        
        for item in data:
            prompt_str = item["prompt"]
            response_str = item["response"]
            
            prompt_ids = tokenizer.encode(prompt_str).ids
            # Append EOS token to the end of the response
            response_ids = tokenizer.encode(response_str).ids + [eos_id]
            
            # Skip samples that exceed the context window
            if len(prompt_ids) + len(response_ids) > max_length:
                continue
                
            self.samples.append((prompt_ids, response_ids))
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        return self.samples[idx]

def code_collate_fn(batch, max_length=512, pad_id=0, ignore_index=-100):
    """
    Collates individual (prompt_ids, response_ids) into padded batched tensors.
    Creates:
      - input_ids: full sequence [prompt + response]
      - labels: [-100 on prompt] + [response ids] + [-100 on padding]
      - attention_mask: 1 for valid tokens, 0 for padding
    """
    batch_size = len(batch)
    # Determine the maximum sequence length in this batch for efficient dynamic padding
    batch_max_len = max(len(p) + len(r) for p, r in batch)
    seq_len = min(batch_max_len, max_length)
    
    input_ids = torch.full((batch_size, seq_len), pad_id, dtype=torch.long)
    labels = torch.full((batch_size, seq_len), ignore_index, dtype=torch.long)
    
    for i, (prompt_ids, response_ids) in enumerate(batch):
        combined_ids = prompt_ids + response_ids
        actual_len = min(len(combined_ids), seq_len)
        prompt_len = min(len(prompt_ids), actual_len)
        
        input_ids[i, :actual_len] = torch.tensor(combined_ids[:actual_len], dtype=torch.long)
        
        # The prompt part is masked with -100; only the response tokens receive loss
        response_slice = combined_ids[prompt_len:actual_len]
        labels[i, prompt_len:actual_len] = torch.tensor(response_slice, dtype=torch.long)
        
    # For autoregressive decoder modeling: input is x = input_ids[:, :-1], target is y = labels[:, 1:]
    x = input_ids[:, :-1]
    y = labels[:, 1:]
    
    return x, y

In [ ]:
MAX_SEQ_LEN = 256
BATCH_SIZE = 16

print("Building train and validation PyTorch datasets...")
train_dataset = CodeInstructionDataset(train_data, tokenizer, max_length=MAX_SEQ_LEN)
val_dataset = CodeInstructionDataset(val_data, tokenizer, max_length=MAX_SEQ_LEN)

print(f"Retained {len(train_dataset)} / {len(train_data)} train samples within max_length={MAX_SEQ_LEN}")
print(f"Retained {len(val_dataset)} / {len(val_data)} val samples within max_length={MAX_SEQ_LEN}")

from functools import partial
collate = partial(code_collate_fn, max_length=MAX_SEQ_LEN, pad_id=PAD_TOKEN_ID, ignore_index=-100)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, num_workers=0, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate, num_workers=0, pin_memory=True)

# Verify a single batch
for x_batch, y_batch in train_dataloader:
    print("\nBatch Verification:")
    print("x_batch shape (input_ids):", x_batch.shape)
    print("y_batch shape (labels):   ", y_batch.shape)
    print("First row labels (showing -100 masking on prompt, token IDs on code):\n", y_batch[0, :40].tolist())
    break

## 5. Decoder Transformer Architecture

We enhance your `DecoderTransformer` architecture with two standard optimizations:
1. **Pre-LayerNorm (Pre-LN):** Normalization is applied *before* self-attention and MLP blocks. This prevents gradient explosion and makes deep transformers much more stable.
2. **Weight Tying:** The token embedding weights are shared with the final linear projection layer (`self.output_layer.weight = self.embedding.weight`). This reduces the model size by ~8-16M parameters, saving VRAM without losing accuracy.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(), # GELU is standard for code & GPT models
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
    def forward(self, x):
        return self.network(x)

class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            d_model, 
            num_heads, 
            dropout=dropout,
            batch_first=True,
        )
        self.feed_forward = FeedForwardNetwork(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, causal_mask=None, padding_mask=None):
        # Pre-LN: LayerNorm before attention
        norm_x = self.norm1(x)
        attn_output, _ = self.self_attn(
            norm_x, norm_x, norm_x,
            attn_mask=causal_mask,
            key_padding_mask=padding_mask,
            need_weights=False
        )
        x = x + self.dropout1(attn_output)
        
        # Pre-LN: LayerNorm before feed forward
        ff_output = self.feed_forward(self.norm2(x))
        x = x + self.dropout2(ff_output)
        return x

class DecoderTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=512, num_layers=4, num_heads=8, d_ff=1024, pad_id=0, max_len=512, dropout=0.1):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.positional_encoding = PositionalEncoding(d_model, max_len=max_len, dropout=dropout)
        
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.output_layer = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight Tying
        self.output_layer.weight = self.embedding.weight

    def forward(self, tgt):
        len_tgt = tgt.size(1)
        # Causal mask: upper triangular elements set to True (masked)
        causal_mask = torch.triu(torch.ones(len_tgt, len_tgt, device=tgt.device, dtype=torch.bool), diagonal=1)
        padding_mask = (tgt == self.pad_id) if self.pad_id is not None else None

        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.positional_encoding(x)

        for layer in self.layers:
            x = layer(x, causal_mask=causal_mask, padding_mask=padding_mask)

        x = self.final_norm(x)
        logits = self.output_layer(x)
        return logits

In [ ]:
vocab_size = tokenizer.get_vocab_size()
model = DecoderTransformer(
    vocab_size=vocab_size,
    d_model=512,
    num_layers=4,
    num_heads=8,
    d_ff=1024,
    pad_id=PAD_TOKEN_ID,
    max_len=MAX_SEQ_LEN,
    dropout=0.1
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model initialized!")
print(f"Total parameters: {total_params / 1e6:.2f}M | Trainable: {trainable_params / 1e6:.2f}M")

## 6. Training Setup & Supervised Fine-Tuning Loop

Key training components:
- `criterion = nn.CrossEntropyLoss(ignore_index=-100)`: Ignores both padded tokens and prompt tokens.
- `torch.amp.autocast('cuda')`: FP16 mixed precision acceleration.
- Cosine Annealing Learning Rate Scheduler.
- Validation loss evaluation after every epoch with automatic checkpoint saving to `best_coding_assistant.pt`.

In [ ]:
NUM_EPOCHS = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-5)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

In [ ]:
def evaluate(model, dataloader, criterion, vocab_size, device):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                logits = model(x)
                loss = criterion(logits.view(-1, vocab_size), y.view(-1))
            
            if not torch.isnan(loss):
                total_loss += loss.item()
                total_batches += 1
                
    return total_loss / max(1, total_batches)

CHECKPOINT_PATH = "best_coding_assistant.pt"
best_val_loss = float("inf")

print(f"Starting SFT Training for {NUM_EPOCHS} epochs on {device}...\n")

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    model.train()
    train_loss = 0.0
    batches_counted = 0

    pbar = tqdm(train_dataloader, desc=f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}]", leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            logits = model(x)
            loss = criterion(logits.view(-1, vocab_size), y.view(-1))

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        if not torch.isnan(loss):
            train_loss += loss.item()
            batches_counted += 1
            pbar.set_postfix(batch_loss=f"{loss.item():.4f}")

    scheduler.step()
    avg_train_loss = train_loss / max(1, batches_counted)
    avg_val_loss = evaluate(model, val_dataloader, criterion, vocab_size, device)
    elapsed = time.time() - start_time

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        saved_indicator = "★ (Saved Best)"
    else:
        saved_indicator = ""

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({elapsed:.1f}s) | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} {saved_indicator}"
    )

print(f"\nTraining complete! Best model saved to {CHECKPOINT_PATH}")

## 7. Interactive Code Generation Function

Now we build `generate_code()` to answer user coding questions:
1. Wraps question in the instruction template: `### Instruction:\n...\n\n### Response:\n`
2. Encodes prompt **without** `[EOS]`
3. Autoregressive loop:
   - Uses **Greedy Decoding** (`argmax`) or **Top-k / Top-p Nucleus Sampling** with low temperature (`0.35`)
   - Stops decoding immediately when the model outputs `[EOS]`
4. Decodes and strips the prompt prefix, returning clean executable Python code.

In [ ]:
def generate_code(
    model,
    tokenizer,
    user_instruction,
    input_text="",
    max_new_tokens=256,
    temperature=0.35,
    top_k=40,
    top_p=0.85,
    device=device,
    repetition_penalty=1.15
):
    """
    Generates code for a given natural language programming question.
    """
    model.eval()
    
    # 1. Format user question into training prompt template
    prompt_str = format_instruction(user_instruction, input_text)
    prompt_tokens = tokenizer.encode(prompt_str).ids
    
    input_ids = torch.tensor(prompt_tokens, dtype=torch.long, device=device).unsqueeze(0)
    prompt_length = input_ids.shape[1]
    eos_id = tokenizer.token_to_id("[EOS]")
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Condition on the last 256 tokens (context window)
            input_cond = input_ids[:, -256:]
            logits = model(input_cond)
            next_token_logits = logits[:, -1, :].clone()

            if repetition_penalty != 1.0:
                generated_tokens = set(input_ids[0, prompt_length:].tolist())
                for prev_token in generated_tokens:
                    if next_token_logits[0, prev_token] < 0:
                        next_token_logits[0, prev_token] *= repetition_penalty
                    else:
                        next_token_logits[0, prev_token] /= repetition_penalty

            # Greedy decoding when temperature is very low
            if temperature < 1e-3:
                next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            else:
                next_token_logits = next_token_logits / temperature
                
                # Top-K filtering
                if top_k > 0:
                    indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][..., -1, None]
                    next_token_logits[indices_to_remove] = -float('Inf')
                    
                # Top-P (Nucleus) filtering
                if top_p < 1.0:
                    sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
                    cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
                    sorted_indices_to_remove = cumulative_probs > top_p
                    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                    sorted_indices_to_remove[..., 0] = 0
                    indices_to_remove = sorted_indices[sorted_indices_to_remove]
                    next_token_logits[:, indices_to_remove] = -float('Inf')
                    
                probs = torch.softmax(next_token_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                
            # Check for EOS early stopping
            if next_token.item() == eos_id:
                break
                
            input_ids = torch.cat([input_ids, next_token], dim=1)
            
    # Extract generated tokens (excluding prompt)
    generated_tokens = input_ids[0, prompt_length:].tolist()
    generated_code = tokenizer.decode(generated_tokens)
    return generated_code.strip()

In [ ]:
# Load the best trained model checkpoint
if os.path.exists(CHECKPOINT_PATH):
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
    print("Loaded best model checkpoint!")

# Test query requested by the user
user_query = "Write a python program to find the maximum number in an array"
print("="*60)
print(f"QUESTION: {user_query}")
print("="*60)

response_code = generate_code(model, tokenizer, user_query)
print("\n--- RAW MARKDOWN OUTPUT ---\n")
print(response_code)
print("\n" + "="*60)

# Render beautifully formatted markdown in Jupyter
from IPython.display import display, Markdown
display(Markdown(f"### Assistant Response:\n{response_code}"))

In [ ]:
# Try a few more coding questions
sample_questions = [
    "Write a python function to check if a string is a palindrome",
    "Write a python function to calculate the factorial of a number using recursion",
    "Write a python function to reverse a linked list"
]

for q in sample_questions:
    print("="*60)
    print(f"QUESTION: {q}")
    print("="*60)
    out = generate_code(model, tokenizer, q, temperature=0.2)
    print(out)
    print()